# 05 - Analysis Comparaison: SPLADE vs ColBERT vs DPR

This notebook compares retrieval quality and runtime for the three algorithms used in this project:

- SPLADE
- ColBERT
- DPR

Metrics reported:

- MRR@10
- Recall@10
- Mean latency (s)
- P95 latency (s)

> Prerequisites: run data preparation and indexing notebooks first so `passages`, `queries`, `qrels`, `splade`, `colbert`, and `dpr` are populated.

In [1]:
import sys
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve().parent
print(f"Project root: {project_root}")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.database.connection import get_connection
from src.splade.search import search_gin
from src.splade.encoder import SpladeEncoder
from src.colbert.search import search_bruteforce as colbert_search
from src.colbert.encoder import ColBERTEncoder
from src.dpr.encode import get_model

Project root: /home/tim/Documents/Projet_Big_Data_IR


/home/tim/Documents/Projet_Big_Data_IR/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Configuration

Keep values small for fast experimentation.

In [2]:
TOP_K = 10
SAMPLE_SIZE = 5
RANDOM_SEED = 42

# ColBERT here is brute-force MaxSim; keep this cap low for practical runtimes.
COLBERT_MAX_PASSAGES = 10_000

print(f"TOP_K={TOP_K}")
print(f"SAMPLE_SIZE={SAMPLE_SIZE}")
print(f"RANDOM_SEED={RANDOM_SEED}")
print(f"COLBERT_MAX_PASSAGES={COLBERT_MAX_PASSAGES}")

TOP_K=10
SAMPLE_SIZE=5
RANDOM_SEED=42
COLBERT_MAX_PASSAGES=10000


## 2) Build Evaluation Query Set

We sample queries that have at least one relevant passage (`relevance=1`).

In [3]:
conn = get_connection()
cursor = conn.cursor()

cursor.execute(
    """
    SELECT DISTINCT q.id, q.text
    FROM queries q
    JOIN qrels qr ON qr.query_id = q.id
    WHERE qr.relevance = 1
    ORDER BY q.id
    """
)
all_queries = cursor.fetchall()

if len(all_queries) == 0:
    cursor.close()
    conn.close()
    raise RuntimeError("No eligible queries found. Check qrels population.")

rng = random.Random(RANDOM_SEED)
if len(all_queries) > SAMPLE_SIZE:
    eval_queries = rng.sample(all_queries, SAMPLE_SIZE)
else:
    eval_queries = all_queries

eval_queries = sorted(eval_queries, key=lambda x: x[0])
eval_query_ids = [q[0] for q in eval_queries]

cursor.execute(
    """
    SELECT query_id, passage_id
    FROM qrels
    WHERE query_id = ANY(%s) AND relevance = 1
    """,
    (eval_query_ids,),
)

relevant_by_query = {}
for query_id, passage_id in cursor.fetchall():
    relevant_by_query.setdefault(query_id, set()).add(passage_id)

cursor.close()
conn.close()

print(f"Eligible queries in DB: {len(all_queries)}")
print(f"Evaluation queries selected: {len(eval_queries)}")
display(pd.DataFrame(eval_queries, columns=["query_id", "query_text"]).head(10))

2026-04-16 14:58:48,570 - INFO - Database connection established successfully.


Eligible queries in DB: 79704
Evaluation queries selected: 5


,query_id,query_text
0,23081,which muscles are used to extend out your stomach
1,34772,What month and day is sweetest day
2,49911,does a kenyan citizen need a transit visa for ...
3,52858,how many pics can i attach to an email
4,56932,how does esperanza try to prepare herself for ...


## 3) Unified Evaluation Helpers

In [4]:
def reciprocal_rank_at_k(retrieved_ids, relevant_ids, k=10):
    for rank, pid in enumerate(retrieved_ids[:k], 1):
        if pid in relevant_ids:
            return 1.0 / rank
    return 0.0


def recall_at_k(retrieved_ids, relevant_ids, k=10):
    if not relevant_ids:
        return 0.0
    hits = len(set(retrieved_ids[:k]) & relevant_ids)
    return hits / len(relevant_ids)


def run_dpr_topk(query_text, dpr_model, conn, top_k=10):
    qemb = dpr_model.encode([query_text], convert_to_numpy=True)[0].tolist()
    cur = conn.cursor()
    cur.execute(
        """
        SELECT d.passage_id
        FROM dpr d
        ORDER BY d.embedding <=> %s::vector
        LIMIT %s
        """,
        (qemb, top_k),
    )
    ids = [row[0] for row in cur.fetchall()]
    cur.close()
    return ids


def evaluate_algorithm(name, queries, relevant_lookup, runner, top_k=10, progress_every=5):
    rows = []
    for i, (query_id, query_text) in enumerate(queries, 1):
        t0 = time.perf_counter()
        retrieved_ids = runner(query_text)
        latency_s = time.perf_counter() - t0

        relevant_ids = relevant_lookup.get(query_id, set())
        rr = reciprocal_rank_at_k(retrieved_ids, relevant_ids, k=top_k)
        rec = recall_at_k(retrieved_ids, relevant_ids, k=top_k)

        rows.append(
            {
                "algorithm": name,
                "query_id": query_id,
                "query_text": query_text,
                "reciprocal_rank": rr,
                "recall_at_k": rec,
                "latency_s": latency_s,
                "returned_docs": len(retrieved_ids),
            }
        )

        if i % progress_every == 0 or i == len(queries):
            print(f"[{name}] {i}/{len(queries)} queries done")

    return pd.DataFrame(rows)

## 4) Run Comparison

This cell may take time, especially for ColBERT.

In [ ]:
conn = get_connection()
try:
    diag_cur = conn.cursor()
    diag_cur.execute("SELECT COUNT(*) FROM passages")
    row = diag_cur.fetchone()
    if row is None:
        raise RuntimeError("Could not read passages count (fetchone returned None).")
    n_passages = int(row[0])

    indexed_counts = {}
    for algo, table in [("splade", "splade"), ("colbert", "colbert"), ("dpr", "dpr")]:
        diag_cur.execute(f"SELECT COUNT(*) FROM {table}")
        row = diag_cur.fetchone()
        if row is None:
            raise RuntimeError(f"Could not read count for table '{table}' (fetchone returned None).")
        indexed_counts[algo] = int(row[0])

    diag_cur.close()

    diag_df = pd.DataFrame(
        [
            {
                "algorithm": k,
                "indexed_rows": v,
                "coverage_vs_passages": (v / n_passages) if n_passages else 0.0,
            }
            for k, v in indexed_counts.items()
        ]
    )
    print(f"Total passages: {n_passages}")
    display(diag_df)

    if indexed_counts["dpr"] == 0:
        raise RuntimeError(
            "DPR table is empty. Run DPR indexing notebook/cell first; otherwise DPR cannot appear in results."
        )

    splade_encoder = SpladeEncoder()
    colbert_encoder = ColBERTEncoder()
    dpr_model = get_model()

    splade_df = evaluate_algorithm(
        name="splade",
        queries=eval_queries,
        relevant_lookup=relevant_by_query,
        top_k=TOP_K,
        runner=lambda q: [
            r["passage_id"]
            for r in search_gin(
                q,
                top_k=TOP_K,
                conn=conn,
                encoder=splade_encoder,
                log_search=False,
            )
        ],
    )

    colbert_df = evaluate_algorithm(
        name="colbert",
        queries=eval_queries,
        relevant_lookup=relevant_by_query,
        top_k=TOP_K,
        runner=lambda q: [
            r["passage_id"]
            for r in colbert_search(
                q,
                top_k=TOP_K,
                max_passages=COLBERT_MAX_PASSAGES,
                conn=conn,
                encoder=colbert_encoder,
                log_search=False,
            )
        ],
    )

    dpr_df = evaluate_algorithm(
        name="dpr",
        queries=eval_queries,
        relevant_lookup=relevant_by_query,
        top_k=TOP_K,
        runner=lambda q: run_dpr_topk(q, dpr_model=dpr_model, conn=conn, top_k=TOP_K),
    )
finally:
    conn.close()

per_query_df = pd.concat([splade_df, colbert_df, dpr_df], ignore_index=True)
per_query_df = per_query_df.sort_values(["algorithm", "query_id"]).reset_index(drop=True)

counts_df = per_query_df.groupby("algorithm", as_index=False).agg(
    queries_evaluated=("query_id", "count"),
    avg_docs_returned=("returned_docs", "mean"),
)

print("Per-algorithm row counts (DPR must be present here):")
display(counts_df)

# Display with rounded latency for readability.
per_query_display_df = per_query_df.copy()
per_query_display_df["latency_s"] = per_query_display_df["latency_s"].round(2)
display(per_query_display_df)

2026-04-16 15:11:51,989 - INFO - Database connection established successfully.


Total passages: 676193


,algorithm,indexed_rows,coverage_vs_passages
0,splade,676193,1.000000
1,colbert,676193,1.000000
2,dpr,600000,0.887321


2026-04-16 15:11:52,251 - INFO - Loading SPLADE model 'naver/splade-cocondenser-ensembledistil' on cpu ...
2026-04-16 15:11:52,399 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-16 15:11:52,413 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/naver/splade-cocondenser-ensembledistil/49cf4c7b0db5b870a401ddf5e2669993ef3699c7/config.json "HTTP/1.1 200 OK"
2026-04-16 15:11:52,530 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-16 15:11:52,541 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/naver/splade-cocondenser-ensembledistil/49cf4c7b0db5b870a401ddf5e2669993ef3699c7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-16 15:11:52,655 - INFO - HTTP Request: GET https://huggingface.co/api/models/naver/splade-cocon

[splade] 5/5 queries done


...

[colbert] 5/5 queries done


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.03it/s]


[dpr] 5/5 queries done
Per-algorithm row counts (DPR must be present here):


,algorithm,queries_evaluated,avg_docs_returned
0,colbert,5,10.0
1,dpr,5,10.0
2,splade,5,10.0


,algorithm,query_id,query_text,reciprocal_rank,recall_at_k,latency_s,returned_docs
0,colbert,23081,which muscles are used to extend out your stomach,0.000,0.0,36.85,10
1,colbert,34772,What month and day is sweetest day,0.000,0.0,45.08,10
2,colbert,49911,does a kenyan citizen need a transit visa for ...,0.000,0.0,43.76,10
3,colbert,52858,how many pics can i attach to an email,0.000,0.0,44.69,10
4,colbert,56932,how does esperanza try to prepare herself for ...,0.000,0.0,41.01,10
5,dpr,23081,which muscles are used to extend out your stomach,0.000,0.0,0.13,10
6,dpr,34772,What month and day is sweetest day,0.000,0.0,0.07,10
7,dpr,49911,does a kenyan citizen need a transit visa for ...,0.000,0.0,0.05,10
8,dpr,52858,how many pics can i attach to an email,0.125,1.0,0.09,10
9,dpr,56932,how does esperanza try to prepare herself for ...,0.000,0.0,0.13,10


## 5) Aggregate Metrics Table

In [9]:
summary_df = (
    per_query_df.groupby("algorithm", as_index=False)
    .agg(
        queries=("query_id", "count"),
        mrr_at_10=("reciprocal_rank", "mean"),
        recall_at_10=("recall_at_k", "mean"),
        mean_latency_s=("latency_s", "mean"),
        p95_latency_s=("latency_s", lambda s: float(np.percentile(s, 95))),
    )
)

expected_order = ["splade", "colbert", "dpr"]
summary_df = summary_df.set_index("algorithm").reindex(expected_order).reset_index()

summary_df[["mrr_at_10", "recall_at_10"]] = summary_df[["mrr_at_10", "recall_at_10"]].round(4)
summary_df[["mean_latency_s", "p95_latency_s"]] = summary_df[["mean_latency_s", "p95_latency_s"]].round(2)

print("Final comparison table:")
display(summary_df)

Final comparison table:


,algorithm,queries,mrr_at_10,recall_at_10,mean_latency_s,p95_latency_s
0,splade,5,0.600,0.6,25.75,31.57
1,colbert,5,0.000,0.0,42.28,45.00
2,dpr,5,0.025,0.2,0.09,0.13


## 6) Optional: One Query Side-by-Side

Use this cell for quick qualitative inspection of returned passages.

In [7]:
example_query = "what is the speed of light"

conn = get_connection()
try:
    splade_encoder = SpladeEncoder()
    colbert_encoder = ColBERTEncoder()
    dpr_model = get_model()

    splade_results = search_gin(
        example_query,
        top_k=3,
        conn=conn,
        encoder=splade_encoder,
        log_search=False,
    )
    colbert_results = colbert_search(
        example_query,
        top_k=3,
        max_passages=COLBERT_MAX_PASSAGES,
        conn=conn,
        encoder=colbert_encoder,
        log_search=False,
    )

    dpr_ids = run_dpr_topk(example_query, dpr_model=dpr_model, conn=conn, top_k=3)
    cur = conn.cursor()
    cur.execute("SELECT id, text FROM passages WHERE id = ANY(%s)", (dpr_ids,))
    dpr_texts = {row[0]: row[1] for row in cur.fetchall()}
    cur.close()

    print(f"Query: {example_query}\n")

    print("SPLADE top-3:")
    for i, r in enumerate(splade_results, 1):
        print(f"  [{i}] {r['score']:.4f} - {r['text'][:120]}...")

    print("\nColBERT top-3:")
    for i, r in enumerate(colbert_results, 1):
        print(f"  [{i}] {r['score']:.4f} - {r['text'][:120]}...")

    print("\nDPR top-3:")
    for i, pid in enumerate(dpr_ids, 1):
        txt = dpr_texts.get(pid, "")
        print(f"  [{i}] passage_id={pid} - {txt[:120]}...")
finally:
    conn.close()

2026-04-16 15:06:18,145 - INFO - Database connection established successfully.
2026-04-16 15:06:18,146 - INFO - Loading SPLADE model 'naver/splade-cocondenser-ensembledistil' on cpu ...
2026-04-16 15:06:18,302 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-16 15:06:18,314 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/naver/splade-cocondenser-ensembledistil/49cf4c7b0db5b870a401ddf5e2669993ef3699c7/config.json "HTTP/1.1 200 OK"
2026-04-16 15:06:18,426 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-16 15:06:18,438 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/naver/splade-cocondenser-ensembledistil/49cf4c7b0db5b870a401ddf5e2669993ef3699c7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-16 15:06:18,608 

Query: what is the speed of light

SPLADE top-3:
  [1] 22.6496 - View full size image. The speed of light in a vacuum is 186,282 miles per second (299,792 kilometers per second), and in...
  [2] 22.3885 - View full size image. The speed of light in a vacuum is 186,282 miles per second (299,792 kilometers per second), and in...
  [3] 21.5042 - The speed of light in a medium is the speed of light in a vacuum divided by the index of refract … ion of the material. ...

ColBERT top-3:
  [1] 9.4745 - Stop using phenazopyridine and call your doctor at once if you have any of these serious side effects: 1  pale skin, fev...
  [2] 8.9279 - Before taking phenazopyridine, tell your doctor or pharmacist if you are allergic to it; or if you have any other allerg...
  [3] 8.5149 - CFCs Chlorofluorocarbon (CFC) is an organic compound that contains carbon, chlorine, and fluorine, produced as a volatil...

DPR top-3:
  [1] passage_id=165548 - The speed (or sometimes you might see it called velocity) of

## Reading the Results (Simple Guide)

### 1) What each metric means

- **MRR@10 (Mean Reciprocal Rank)**
  - Question answered: *"How early does the first good result appear?"*
  - Higher is better.
  - A high MRR means users often see a relevant passage in the first positions.

- **Recall@10**
  - Question answered: *"How many relevant passages are found in the top 10?"*
  - Higher is better.
  - A high Recall means the system misses fewer relevant passages.

- **Mean latency (s)**
  - Question answered: *"How long does a search take on average?"*
  - Lower is better.
  - This is the average speed users feel over many queries.

- **P95 latency (s)**
  - Question answered: *"How slow are the worst normal cases?"*
  - Lower is better.
  - If P95 is much higher than mean latency, response time is unstable.

### 2) How to compare algorithms quickly

- If one model has better MRR/Recall and similar latency, it is usually the best choice.
- If quality is similar, choose the one with lower mean and lower P95 latency.
- If one model has great quality but very high P95 latency, it may feel slow in production.

### 3) Final conclusion: pros and cons of each IR method

- **SPLADE**
  - Pros: usually strong lexical precision, interpretable term-based behavior, often a good quality/speed trade-off with sparse indexing.
  - Cons: can miss semantic matches when wording is very different.

- **ColBERT**
  - Pros: can have very high relevance quality thanks to late interaction.
  - Cons: expensive computing and memory usage. Much slower without ANN or heavy optimization.

- **DPR**
  - Pros: simple dense retrieval pipeline, fast nearest-neighbor style search, good semantic matching.
  - Cons: can be weaker on exact lexical constraints and sensitive to embedding model/domain mismatch.

### 4) Practical decision rule

- Balanced quality/speed scenario: often **SPLADE**.
- Speed-first and simple deployment scenario: often **DPR**.